# Phan tich binh luan YouTube

Notebook nay gom 7 phan: import thu vien, doc du lieu raw, kiem tra chat luong du lieu, lam sach du lieu, EDA, gan nhan sentiment va export CSV cuoi.

## 1. Import thu vien

In [ ]:
# Thu vien chuan
import re
import string
import unicodedata
from collections import Counter
from pathlib import Path

# Xu ly du lieu
import numpy as np
import pandas as pd

# Truc quan hoa
import matplotlib.pyplot as plt
import seaborn as sns

# Thiet lap hien thi
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.unicode_minus"] = False

# Duong dan du lieu
RAW_DATA_PATH = Path("youtube_comments_raw.csv")
FINAL_DATA_PATH = Path("youtube_comments_clean.csv")

## 2. Doc du lieu raw

In [ ]:
# Doc du lieu goc
df = pd.read_csv(RAW_DATA_PATH)

print(f"So dong: {df.shape[0]:,}")
print(f"So cot: {df.shape[1]:,}")
df.head()

In [ ]:
# Thong tin kieu du lieu
df.info()

## 3. Data Quality Check

In [ ]:
# Kiem tra NULL theo tung cot
null_summary = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "null_count": df.isna().sum().values,
    "null_percent": (df.isna().mean() * 100).round(2).values,
})

null_summary

In [ ]:
# Kiem tra comment rong va duplicate
empty_comment_count = int((df["comment_text"].isna() | (df["comment_text"].astype("string").str.strip() == "")).sum())

quality_summary = pd.DataFrame([
    {"check": "So dong co NULL", "count": int(df.isna().any(axis=1).sum())},
    {"check": "Comment rong/NULL", "count": empty_comment_count},
    {"check": "Author thieu", "count": int(df["author"].isna().sum())},
    {"check": "Duplicate tat ca cot", "count": int(df.duplicated().sum())},
    {"check": "Duplicate video_id + comment_id", "count": int(df.duplicated(subset=["video_id", "comment_id"]).sum())},
    {"check": "Duplicate comment_text", "count": int(df.duplicated(subset=["comment_text"]).sum())},
])

quality_summary

In [ ]:
# Xem cac dong co NULL
rows_with_null = df[df.isna().any(axis=1)]
rows_with_null

In [ ]:
# Xem cac comment bi trung theo video_id + comment_id
duplicate_raw = df[df.duplicated(subset=["video_id", "comment_id"], keep=False)]
duplicate_raw.sort_values(["video_id", "comment_id"])

## 4. Data Cleaning

In [ ]:
# Tao ban sao de lam sach
df_clean = df.copy()

# Xoa comment_text rong/NULL
empty_comment_mask = df_clean["comment_text"].isna() | (df_clean["comment_text"].astype("string").str.strip() == "")
removed_empty_comments = int(empty_comment_mask.sum())
df_clean = df_clean.loc[~empty_comment_mask].copy()

# Fill author thieu
missing_author_count = int(df_clean["author"].isna().sum())
df_clean["author"] = df_clean["author"].fillna("unknown_author")

# Xoa duplicate theo khoa comment duy nhat
before_dedup = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=["video_id", "comment_id"], keep="first").reset_index(drop=True)
removed_duplicate_comments = before_dedup - len(df_clean)

print(f"So comment rong/NULL da xoa: {removed_empty_comments:,}")
print(f"So author thieu da fill: {missing_author_count:,}")
print(f"So duplicate da xoa: {removed_duplicate_comments:,}")
print(f"So dong sau khi lam sach: {len(df_clean):,}")

In [ ]:
# Chuan hoa thoi gian va tao cot ngay thang
df_clean["published_at"] = pd.to_datetime(df_clean["published_at"], errors="coerce")
df_clean["updated_at"] = pd.to_datetime(df_clean["updated_at"], errors="coerce")

df_clean["date"] = df_clean["published_at"].dt.date
df_clean["year"] = df_clean["published_at"].dt.year
df_clean["month"] = df_clean["published_at"].dt.month
df_clean["day"] = df_clean["published_at"].dt.day
df_clean["hour"] = df_clean["published_at"].dt.hour

df_clean[["published_at", "date", "year", "month", "day", "hour"]].head()

In [ ]:
# Tao cot clean_text de phuc vu EDA va sentiment labeling
def clean_comment_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df_clean["clean_text"] = df_clean["comment_text"].apply(clean_comment_text)
df_clean[["comment_text", "clean_text"]].head()

In [ ]:
# Xem cac dong co clean_text rong sau khi lam sach text
blank_clean_text_mask = df_clean["clean_text"].astype("string").str.strip() == ""
blank_clean_text_rows = df_clean.loc[blank_clean_text_mask].copy()

print(f"So dong co clean_text rong: {len(blank_clean_text_rows):,}")
blank_clean_text_rows[["video_id", "comment_id", "author", "comment_text", "clean_text", "like_count", "reply_count"]]

In [ ]:
# Xoa cac dong co clean_text rong
before_remove_blank_text = len(df_clean)
df_clean = df_clean.loc[~blank_clean_text_mask].reset_index(drop=True)
removed_blank_clean_text = before_remove_blank_text - len(df_clean)

print(f"So dong clean_text rong da xoa: {removed_blank_clean_text:,}")
print(f"So dong sau khi xoa clean_text rong: {len(df_clean):,}")
print(f"clean_text rong con lai: {(df_clean['clean_text'].astype('string').str.strip() == '').sum():,}")

In [ ]:
# So sanh truoc va sau khi lam sach
comparison_summary = pd.DataFrame({
    "metric": [
        "So dong",
        "Tong gia tri NULL",
        "So dong co NULL",
        "Comment rong/NULL",
        "Author thieu",
        "Clean text rong",
        "Duplicate video_id + comment_id",
    ],
    "before_cleaning": [
        len(df),
        int(df.isna().sum().sum()),
        int(df.isna().any(axis=1).sum()),
        int((df["comment_text"].isna() | (df["comment_text"].astype("string").str.strip() == "")).sum()),
        int(df["author"].isna().sum()),
        removed_blank_clean_text,
        int(df.duplicated(subset=["video_id", "comment_id"]).sum()),
    ],
    "after_cleaning": [
        len(df_clean),
        int(df_clean.isna().sum().sum()),
        int(df_clean.isna().any(axis=1).sum()),
        int((df_clean["comment_text"].isna() | (df_clean["comment_text"].astype("string").str.strip() == "")).sum()),
        int(df_clean["author"].isna().sum()),
        int((df_clean["clean_text"].astype("string").str.strip() == "").sum()),
        int(df_clean.duplicated(subset=["video_id", "comment_id"]).sum()),
    ],
})

comparison_summary["change"] = comparison_summary["after_cleaning"] - comparison_summary["before_cleaning"]
comparison_summary

## 5. Exploratory Data Analysis - EDA

In [ ]:
# Thong ke tong quan cac bien so
df_clean[["like_count", "reply_count"]].describe()

In [ ]:
# So comment theo video
comments_by_video = df_clean["video_id"].value_counts().reset_index()
comments_by_video.columns = ["video_id", "comment_count"]
comments_by_video

In [ ]:
# Bieu do so comment theo video
plt.figure(figsize=(12, 6))
sns.barplot(data=comments_by_video, x="video_id", y="comment_count")
plt.title("So luong comment theo video")
plt.xlabel("Video ID")
plt.ylabel("So comment")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Top comment co nhieu like nhat
top_liked_comments = df_clean.sort_values("like_count", ascending=False).head(10)
top_liked_comments[["video_id", "author", "comment_text", "like_count", "reply_count", "published_at"]]

In [ ]:
# Xu huong so comment theo ngay
comments_by_date = df_clean.groupby("date").size().reset_index(name="comment_count")
comments_by_date["date"] = pd.to_datetime(comments_by_date["date"])

plt.figure(figsize=(12, 6))
sns.lineplot(data=comments_by_date, x="date", y="comment_count")
plt.title("So luong comment theo ngay")
plt.xlabel("Ngay")
plt.ylabel("So comment")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Top tu xuat hien nhieu nhat trong clean_text
vietnamese_stopwords = {
    "la", "va", "co", "cua", "cho", "mot", "nhung", "cac", "thi", "ma", "da", "duoc", "voi", "trong", "nay",
    "minh", "ban", "anh", "em", "chi", "nguoi", "nha", "qua", "lam", "roi", "nhe", "nha", "luon", "that",
}

tokens = " ".join(df_clean["clean_text"]).split()
tokens = [token for token in tokens if len(token) > 1 and token not in vietnamese_stopwords]
top_words = pd.DataFrame(Counter(tokens).most_common(30), columns=["word", "count"])
top_words

## 6. Sentiment Labeling

In [ ]:
# Gan nhan sentiment theo 3 nhom: khen, che, con lai
praise_keywords = {
    "hay", "hay qua", "qua hay", "hay vai", "hay vãi", "hay thật", "hay that", "cuốn", "cuon",
    "đỉnh", "dinh", "đỉnh quá", "dinh qua", "đỉnh vãi", "dinh vai", "xuất sắc", "xuat sac",
    "tuyệt", "tuyet", "tuyệt vời", "tuyet voi", "đáng yêu", "dang yeu", "dễ thương", "de thuong",
    "vui", "vui quá", "vui qua", "hài", "hai", "hài hước", "hai huoc", "cười xỉu", "cuoi xiu",
    "ngon", "giỏi", "gioi", "thích", "thich", "yêu", "yeu", "mê", "me", "hâm mộ", "ham mo",
    "ok", "oke", "ổn", "on", "chất", "chat", "xịn", "xin", "đẹp", "dep", "haha", "hehe",
}

criticism_keywords = {
    "dở", "do", "tệ", "te", "chán", "chan", "nhảm", "nham", "không hay", "khong hay", "ko hay", "k hay",
    "không thích", "khong thich", "ko thích", "ko thich", "ghét", "ghet", "thất vọng", "that vong",
    "kém", "kem", "xấu", "xau", "tệ quá", "te qua", "chán quá", "chan qua", "dở quá", "do qua",
    "vô duyên", "vo duyen", "xàm", "xam", "xàm quá", "xam qua", "rác", "rac", "lố", "lo",
    "cay", "tức", "tuc", "bực", "buc", "khó chịu", "kho chiu", "bất công", "bat cong", "lừa", "lua",
}

def count_keyword_matches(text, keywords):
    text = str(text).lower()
    padded_text = f" {text} "
    tokens = set(text.split())
    score = 0

    for keyword in keywords:
        keyword = keyword.lower().strip()
        if " " in keyword:
            score += int(f" {keyword} " in padded_text)
        else:
            score += int(keyword in tokens)

    return score

def label_sentiment(text):
    praise_score = count_keyword_matches(text, praise_keywords)
    criticism_score = count_keyword_matches(text, criticism_keywords)

    if praise_score > criticism_score:
        return "tich_cuc"
    if criticism_score > praise_score:
        return "tieu_cuc"
    return "trung_binh"

df_clean["sentiment"] = df_clean["clean_text"].apply(label_sentiment)
df_clean["sentiment"].value_counts()

In [ ]:
# Bieu do phan bo sentiment
sentiment_counts = df_clean["sentiment"].value_counts().reset_index()
sentiment_counts.columns = ["sentiment", "count"]

plt.figure(figsize=(8, 5))
sns.barplot(data=sentiment_counts, x="sentiment", y="count", order=["tich_cuc", "trung_binh", "tieu_cuc"])
plt.title("Phan bo sentiment")
plt.xlabel("Sentiment")
plt.ylabel("So comment")
plt.tight_layout()
plt.show()

In [ ]:
# Xem mau comment theo tung sentiment
sample_sentiment_comments = (
    df_clean.sort_values("like_count", ascending=False)
    .groupby("sentiment", group_keys=False)
    .head(3)
)

sample_sentiment_comments[["sentiment", "author", "comment_text", "like_count"]]

## 7. Export final CSV

In [ ]:
# Export du lieu da lam sach va da gan nhan sentiment
try:
    export_path = FINAL_DATA_PATH
    df_clean.to_csv(export_path, index=False, encoding="utf-8-sig")
except PermissionError:
    export_path = FINAL_DATA_PATH.with_name("youtube_comments_clean_final.csv")
    df_clean.to_csv(export_path, index=False, encoding="utf-8-sig")

print(f"Da export file: {export_path}")
print(f"So dong: {len(df_clean):,}")
print(f"So cot: {df_clean.shape[1]:,}")